# Medicaid Providers List

This takes the raw data file from the Missouri Department of Social Services most recent active providers list.

Data updated 2/14/2026 and run date: 2/14/2026

## Data Provenance:
Missouri Family Support Division1
Provider Enrollment Unit
MMIS ACTIVE PROVIDERS IN MISSOURI AND BORDERING STA20
https://apps.dss.mo.gov/fmsMedicaidProviderSearch/ProviderReports.aspx26

The original data can have multiple rowsNPIntity basethe practice locations and effective date. Per MFCU subject matter expertise, the data reporting should not be based on the effective ds. s.##  

Data Cleaningeps
 St- Show the states within the data
- 
epRemove all columns except for NPI
- Remove duplicates so that it is a single NPI lise)

Return clean dataset NPIslid20
/dsa/groups/casestudycf/tebronze/active_ffs_medicaid_vidproers_bro.."
csv

In [1]:
# Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Change all headers to snake case
import re

def to_snake_case(name: str) -> str:
    # Add underscore between lower-to-upper transitions
    name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)

    # Replace non-alphanumeric with underscores
    name = re.sub(r'[^0-9a-zA-Z]+', '_', name)

    # Remove leading/trailing underscores and lowercase
    return name.strip("_").lower()

In [3]:
# Read in CSV from file path
df = pd.read_csv("/dsa/groups/casestudycf25/team02/bronze/active_ffs_medicaid_providers.csv")

df = df.rename(columns={col: to_snake_case(col) for col in df.columns})

df.head(5)

,npi,provider_name,practice_address,city,state,zip_code,effective_date,phone_number
0,1972567923,,"INSTITUTE FOR FAMILY MEDICINE, 200 CHURCH STREET",ST LOUIS,MO,63135-2413,08/01/2002,(314) 824-2043
1,1033766035,,"FIRESIDE HEALING, 1335 E REPUBLIC ROAD SUITE H",SPRINGFIELD,MO,65804,01/01/2020,(417) 719-1440
2,1013917301,,"AMERICAN DIAGNOSTICS SERVICES LLC, 11012 LINVA...",ST LOUIS,MO,63123-7217,12/22/2020,(443) 662-4101
3,1013917301,,"REONO BERTAGNOLLI A MEDICAL GROUP, 11012 LINVA...",ST LOUIS,MO,63123-7217,12/22/2020,(443) 662-4101
4,1467447607,,"PHYSICIAN GROUPS, LC, 1520 WENTZVILLE PKWY",WENTZVILLE,MO,63385,08/01/2010,(636) 497-4000


In [4]:
print(f"Original Dataset Shape: {df.shape}")

Original Dataset Shape: (178379, 8)


In [5]:
##### LIST ALL STATES INCLUDES MISSOURI AND BORDERING STATES
df['state'].unique()

array(['MO', 'IL', 'IA', 'OK', 'AR', 'TN', 'KS', 'NE', 'KY'], dtype=object)

In [6]:
# Convert npi
df['npi'] = df['npi'].replace(0, np.nan)

In [7]:
df.replace("", np.nan, inplace=True)

In [8]:
df.dtypes

npi                 object
provider_name       object
practice_address    object
city                object
state               object
zip_code            object
effective_date      object
phone_number        object
dtype: object

In [9]:
df_only_npi = pd.DataFrame(df['npi'].drop_duplicates())

In [24]:
df_only_npi.dropna(inplace = True)

In [25]:
df_only_npi.reset_index(drop=True, inplace=True)

In [26]:
print(f"New Dataset Shape After Filtering for Valid NPIs: {df_only_npi.shape}")

New Dataset Shape After Filtering for Valid NPIs: (70169, 1)


In [27]:
df_only_npi.head()

,npi
0,1972567923
1,1033766035
2,1013917301
3,1467447607
4,1013452861


In [28]:
##############################################################
# Return a dataset that is clean for modeling use
##############################################################

df_only_npi.to_csv("/dsa/groups/casestudycf25/team02/silver/active_ffs_medicaid_providers.csv", index=False)
df_only_npi.shape

(70169, 1)

In [30]:
# Check the newly created CSV
temp = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/active_ffs_medicaid_providers.csv", skip_blank_lines=True)

temp.shape

(70169, 1)

In [33]:
temp.head()

,npi
0,1972567923
1,1033766035
2,1013917301
3,1467447607
4,1013452861


In [16]:
only_in_df1 = df_only_npi.merge(temp, how='outer', indicator=True).query('_merge == "left_only"').drop('_merge', axis=1)
only_in_df2 = df_only_npi.merge(temp, how='outer', indicator=True).query('_merge == "right_only"').drop('_merge', axis=1)

anti_union_alt = pd.concat([only_in_df1, only_in_df2], ignore_index=True)

In [17]:
anti_union_alt

,npi
0,
